In [1]:
#Clonamos el proyecto
!git clone https://github.com/Pulido325051/ACTIVIDAD-5.git
%cd ACTIVIDAD-5
!pip install -q mlflow

Cloning into 'ACTIVIDAD-5'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 10 (delta 1), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (1/1), done.
/content/ACTIVIDAD-5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import sys
sys.path.append("fuentes")

import hashlib
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

from datos_prep import cargar_datos, limpiar_datos, dividir_datos

In [3]:
df = cargar_datos()
print(df.shape)
df.head()

(178, 14)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [4]:
#Se revisa la calidad de los datos
print(df.isnull().sum().sum(), "valores nulos")
print(df.duplicated().sum(), "filas duplicadas")
print(df["target"].value_counts())

0 valores nulos
0 filas duplicadas
target
1    71
0    59
2    48
Name: count, dtype: int64


In [5]:
#limpiamos y guardamos
df_limpio = limpiar_datos(df)
print("Antes:", df.shape, "Después:", df_limpio.shape)

Antes: (178, 14) Después: (178, 14)


In [6]:
#Se divide el entrenamiento y prueba
X_train, X_test, y_train, y_test = dividir_datos(df_limpio)
print(X_train.shape, X_test.shape)

(142, 13) (36, 13)


In [7]:
#Identificamos la version de los datos
with open("datos/datos_limp.csv", "rb") as f:
    hash_datos = hashlib.md5(f.read()).hexdigest()
print(hash_datos)

2925c4a8cd201c79d97d3d283429ef23


In [8]:
#Se definen los dos algoritmos
modelos = {
    "regresion_logistica": (
        Pipeline([("escalador", StandardScaler()),
                  ("modelo", LogisticRegression(max_iter=1000))]),
        {"modelo__C": [0.01, 0.1, 1, 10]},
    ),
    "random_forest": (
        RandomForestClassifier(random_state=42),
        {"n_estimators": [50, 100], "max_depth": [3, 5, None]},
    ),
}

In [9]:
#Entrenamos, validamos y registramos en MLflow
mlflow.set_experiment("actividad5_wine")
mejores = {}

for nombre, (modelo, parametros) in modelos.items():
    with mlflow.start_run(run_name=nombre):
        busqueda = GridSearchCV(modelo, parametros, cv=5, scoring="f1_weighted")
        busqueda.fit(X_train, y_train)

        mejor = busqueda.best_estimator_
        pred = mejor.predict(X_test)
        mejores[nombre] = mejor

        mlflow.log_param("algoritmo", nombre)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("hash_datos", hash_datos)
        mlflow.log_params(busqueda.best_params_)

        mlflow.log_metric("cv_f1", busqueda.best_score_)
        mlflow.log_metric("accuracy", accuracy_score(y_test, pred))
        mlflow.log_metric("precision", precision_score(y_test, pred, average="weighted"))
        mlflow.log_metric("recall", recall_score(y_test, pred, average="weighted"))
        mlflow.log_metric("f1", f1_score(y_test, pred, average="weighted"))

        pd.DataFrame(busqueda.cv_results_).to_csv("cv_results.csv", index=False)
        mlflow.log_artifact("cv_results.csv")
        mlflow.log_artifact("datos/datos_limp.csv")
        mlflow.sklearn.log_model(mejor, "modelo", serialization_format="cloudpickle")

        print(nombre, busqueda.best_params_, round(busqueda.best_score_, 3))

2026/10/12 16:58:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/12 16:58:36 INFO mlflow.store.db.utils: Updating database tables
2026/10/12 16:58:41 INFO mlflow.tracking.fluent: Experiment with name 'actividad5_wine' does not exist. Creating a new experiment.
2026/10/12 16:58:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/12 16:58:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


regresion_logistica {'modelo__C': 0.1} 0.993


2026/10/12 16:58:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/12 16:58:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random_forest {'max_depth': 3, 'n_estimators': 50} 0.986


In [10]:
#Tabla comparativa
resultados = mlflow.search_runs(experiment_names=["actividad5_wine"])
columnas = ["tags.mlflow.runName", "metrics.cv_f1", "metrics.accuracy",
            "metrics.precision", "metrics.recall", "metrics.f1"]
resultados[columnas]

,tags.mlflow.runName,metrics.cv_f1,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1
0,random_forest,0.985974,1.0,1.0,1.0,1.0
1,regresion_logistica,0.993183,1.0,1.0,1.0,1.0


In [11]:
#Reporte por clase
for nombre, modelo in mejores.items():
    print("=====", nombre)
    print(classification_report(y_test, modelo.predict(X_test)))

===== regresion_logistica
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

===== random_forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



## Conclusiones

Se compararon dos algoritmos de clasificación, regresión logística y random forest, sobre el dataset Wine (178 muestras, 13 variables, 3 clases). Para cada uno se ajustaron hiperparámetros con GridSearchCV y validación cruzada de 5 folds, usando F1 ponderado como métrica, y todos los experimentos se registraron en MLflow.

En validación cruzada la regresión logística obtuvo un F1 de 0.993 (C = 0.1) y el random forest de 0.986 (max_depth = 3, n_estimators = 50). En el conjunto de prueba ambos modelos alcanzaron 1.00 en accuracy, precision, recall y F1.

La diferencia entre los dos modelos es muy pequeña. Además, el conjunto de prueba tiene solo 36 muestras, por lo que un resultado perfecto no es una evidencia fuerte, y Wine es un dataset pequeño y de clases bien separadas. Por eso elegiría la regresión logística: obtuvo un F1 ligeramente mayor en validación cruzada y es un modelo más simple y fácil de interpretar. Para una comparación más confiable convendría un dataset más grande o más difícil.